**Problem 1**

You're commanding a differential drive robot. When you push the joystick, it accelerates and reaches top speed. The inertia makes the robot slightly tip, its front wheels might get off the ground for a split second, and it might slip a bit. When you remove the joystick, the robot grinds to a halt, but it drifts a tiny bit compared to when you released the joystick.

*Identification step*

We instantly see that the inertia is doing all of this. At the start, acceleration changes, and the robot jerks violently. At the end, it rapidly decelerates, but inertia keeps it going.

As per Newton's second law, inertia is described using the equation $ F = ma $. Since the robot's mass is fixed, reducing the commanded acceleration reduces the force that the drivetrain must generate, resulting in smoother motion.

We then propose a controller as follows:
$$ 
a = \text{clamp}(v_{target} - v_{current}, -v_{lim}, v_{lim})
$$
where
$\text{clamp}(x, a, b)$ limits $x$ to the interval $[a,b]$. We can implement it as follows:

In [ ]:
def clamp(val, min_val, max_val):
    if val < min_val:
        return min_val
    elif val > max_val:
        return max_val
    else:
        return val

print(clamp(-4, 1, 10)) 

Rather than commanding acceleration directly, we gradually update the commanded velocity every control loop.. Instead, it updates the desired velocity every control loop. Since our controller only runs every 20 ms, we don't command a continuous acceleration. Instead, we update the velocity in small discrete steps.
$$ F = m \dfrac{\Delta v}{\Delta t}$$
where 
$$ \Delta v = v_{t} - v_{t-1} $$

The robot then has to remember its previous state, which is $v_{t-1}$. Assuming we already know $dt$ from the control loop time, we can write a class as follows:

In [ ]:
LOOP_DELAY = 20  # milliseconds

class SlewLimiter1:
    def __init__(self, accel):
        self.accel = accel
        self.prev_velocity = 0

    def update(self, input_value):
        max_accel = self.accel * LOOP_DELAY / 1000
        delta = input_value - self.prev_velocity
        delta = clamp(delta, -max_accel, max_accel)

        self.prev_velocity = self.prev_velocity + delta
        return self.prev_velocity

But this has a few edge cases: The most alarming one is that it makes sudden reversal or any change in direction incredibly sluggish. The second one is that we normally want more responsive brakes than acceleration. Acceleration feels laggy if it is slow, and deceleration if matches deceleration makes the stopping feel drifty. We can change this by setting two different limits:
$$ \Delta v = \text{clamp}(\Delta v, -\text{limit}, \text{limit})$$
and for the reversal direction, we can remember it happens only when $ \vec{v} \downarrow \downarrow \vec{a}$. Notice that reversing direction first requires slowing the robot to zero before accelerating in the opposite direction. Physically, this is much closer to braking than accelerating, so we'll use the deceleration limit. Then the code is simply:

In [ ]:
LOOP_DELAY = 20  # milliseconds

class SlewLimiter2:
    def __init__(self, accel, decel):
        self.accel = accel
        self.decel = decel
        self.prev_velocity = 0

    def update(self, input_value):
        delta = input_value - self.prev_velocity
        
        sameDirection = input_value * self.prev_velocity >= 0
        increasingMagnitude = abs(input_value) > abs(self.prev_velocity)

        limit = self.accel if sameDirection and increasingMagnitude else self.decel

        delta = clamp(delta, -limit * LOOP_DELAY / 1000, limit * LOOP_DELAY / 1000)

        self.prev_velocity = self.prev_velocity + delta
        return self.prev_velocity

This gives us smooth acceleration as well as responsive deceleration and reversal, while ensuring that the robot doesn't jerk around. This is called **Slew Rate Limiting**. A slew limiter is model-free: it shapes the command signal without requiring a mathematical model of the robot's dynamics: it limits the acceleration of the actuator, thus ensuring smooth motion. 

For applications with low weight, this is enough. For larger robots or systems carrying heavy payloads, we want acceleration to be smooth also. A sudden change in acceleration means a sudden change in force, and that can cause vibrations, slippage, structural flex, or gearbox shock. To make the motion smooth in tele operation, a better way is to limit the change in acceleration, or **jerk**. The math is similar:

In [ ]:
class JerkLimitedSlew1:
    def __init__(self, max_accel, max_decel, max_jerk):
        self.max_accel = max_accel   
        self.max_decel = max_decel   
        self.max_jerk = max_jerk     
        self.velocity = 0
        self.accel = 0

    def update(self, target_velocity):
        dt = LOOP_DELAY / 1000

        desired_accel = (target_velocity - self.velocity) / dt

        sameDirection = target_velocity * self.velocity >= 0
        increasingMagnitude = abs(target_velocity) > abs(self.velocity)
        accel_limit = self.max_accel if sameDirection and increasingMagnitude else self.max_decel

        desired_accel = clamp(desired_accel, -accel_limit, accel_limit)

        max_delta_accel = self.max_jerk * dt
        accel_delta = clamp(desired_accel - self.accel, -max_delta_accel, max_delta_accel)
        self.accel += accel_delta

        self.velocity += self.accel * dt
        return self.velocity

The idea carries over: we just add the step of calculating 
$$\text{clamp}(\dfrac{da}{dt}, -j_{max}dt, j_{max}dt)$$
Integrating the limited acceleration over time produces the commanded velocity sent to the drivetrain.

Slew and jerk limiters are most commonly used to smooth operator commands during tele-operation. Autonomous systems typically generate trajectories using motion planners that explicitly satisfy velocity, acceleration, and jerk constraints. But still, there is a problem concerning mathematical accuracy in our current implementation: the loop delay is fixed. Usually, letting the teleop loop run at 50Hz with a constant `LOOP_DELAY` like that is good enough. But the signal from the user might have latency, and certain electrical failures can reduce this accuracy. That's why it's better to keep a timer elsewhere, calculate `dt` every loop, and pass it inside our limiter. The code is like so:

In [ ]:
class JerkLimitedSlew2:
    def __init__(self, dt, max_accel, max_decel, max_jerk):
        self.dt = dt
        self.max_accel = max_accel   
        self.max_decel = max_decel   
        self.max_jerk = max_jerk     
        self.velocity = 0
        self.accel = 0

    def update(self, target_velocity):
        desired_accel = (target_velocity - self.velocity) / self.dt

        sameDirection = target_velocity * self.velocity >= 0
        increasingMagnitude = abs(target_velocity) > abs(self.velocity)
        accel_limit = self.max_accel if sameDirection and increasingMagnitude else self.max_decel

        desired_accel = clamp(desired_accel, -accel_limit, accel_limit)

        max_delta_accel = self.max_jerk * self.dt
        accel_delta = clamp(desired_accel - self.accel, -max_delta_accel, max_delta_accel)
        self.accel += accel_delta

        self.velocity += self.accel * self.dt
        return self.velocity